In [ ]:
# full_crawler_batdongsan_hcm.py
# Cào HTML thật qua Cloudflare + phân tích dữ liệu + xuất CSV

import asyncio, random, re, pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

import os

async def crawl_html(url, outfile):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox","--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(
            viewport={"width":1920,"height":1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            locale="vi-VN", timezone_id="Asia/Ho_Chi_Minh")
        await context.add_init_script("""Object.defineProperty(navigator,'webdriver',{get:()=>false});
        window.chrome={runtime:{},};Object.defineProperty(navigator,'languages',{get:()=>['vi-VN','vi','en-US','en']});
        Object.defineProperty(navigator,'plugins',{get:()=>[1,2,3,4,5]});""")

        page = await context.new_page()
        await page.goto(url, wait_until="domcontentloaded")
        await page.mouse.move(random.randint(100,800), random.randint(100,600))
        for y in [400,800,1200]: await page.evaluate(f"window.scrollBy(0,{y})"), await asyncio.sleep(1)
        try: await page.wait_for_load_state("networkidle", timeout=10000) # delay 10s
        except: pass
        await page.wait_for_selector("text=Hiện có", timeout=30000)
        html = await page.content()
        with open(outfile,"w",encoding="utf-8") as f: f.write(html)
        await browser.close()

def analyze_html(infile):
    with open(infile,"r",encoding="utf-8") as f: soup = BeautifulSoup(f,"html.parser")
    cards = soup.select('.js__card-full-web')
    data=[]
    for c in cards:
        try:
            link_tag=c.select_one('a.js__product-link-for-product-id')
            link="https://batdongsan.com.vn"+link_tag['href'] if link_tag else ""
            pid=link_tag['data-product-id'] if link_tag else ""
            title=c.select_one('.re__card-title').get_text(strip=True)
            price=(c.select_one('.re__card-config-price') or {}).get_text(strip=True) if c.select_one('.re__card-config-price') else "Thỏa thuận"
            area_text=(c.select_one('.re__card-config-area') or {}).get_text(strip=True)
            area=re.search(r'\d+\.?\d*',area_text.replace(',',''))
            area=float(area.group()) if area else 0
            bed=(c.select_one('.re__card-config-bedroom span') or {}).get_text(strip=True) if c.select_one('.re__card-config-bedroom span') else "0"
            wc=(c.select_one('.re__card-config-toilet span') or {}).get_text(strip=True) if c.select_one('.re__card-config-toilet span') else "0"
            addr=(c.select_one('.re__card-location') or {}).get_text(strip=True).replace('·','').strip()
            desc=(c.select_one('.re__card-description') or {}).get_text(strip=True)[:400]
            phone=(c.select_one('.hidden-mobile') or {}).get_text(strip=True) if c.select_one('.hidden-mobile') else ""
            if not phone:
                m=re.search(r'0\d{2}\s?\d{3}\s?\*+\*+\*', title+desc)
                phone=m.group() if m else ""
            time_tag=c.select_one('time, .re__card-published-info-published-at')
            date=time_tag.get('title') or time_tag.get_text(strip=True) if time_tag else "Hôm nay"
            agent=(c.select_one('.agent-name') or {}).get_text(strip=True) if c.select_one('.agent-name') else "Không rõ"
            img=(c.select_one('.entry-agent-avatar img') or {}).get('src') if c.select_one('.entry-agent-avatar img') else ""
            photos=(c.select_one('.re__card-image-feature span') or {}).get_text(strip=True) if c.select_one('.re__card-image-feature span') else "0"
            vip=c.get('class',[])
            vip="Diamond" if 'vip-diamond' in vip else "Gold" if 'vip-gold' in vip else "Silver" if 'vip-silver' in vip else "Thường"
            verified="Xác thực" if c.select_one('.re__card-verified') else "Thường"
            pro="Pro" if c.select_one('.re__card-listing-pro-agent-badge') else "Cá nhân"
            data.append([title,price,area,area_text,bed,wc,addr,desc,phone,date,agent,img,photos,vip,verified,pro,link,pid])
        except: continue

    df=pd.DataFrame(data,columns=["Tiêu đề","Giá","Diện tích (m²)","Diện tích","PN","WC","Địa chỉ","Mô tả","SĐT","Ngày đăng","Người đăng","Ảnh đại diện","Số ảnh","Loại tin","Xác thực","Môi giới","Link","ID tin"])
    def per_m2(r):
        if 'tỷ' in r['Giá']:
            try:
                p=float(re.search(r'\d+\.?\d*',r['Giá']).group())
                return round(p*1000/r['Diện tích (m²)'],1) if r['Diện tích (m²)']>0 else 0
            except: return 0
        return 0
    df['Giá/m² (triệu)']=df.apply(per_m2,axis=1)
    df=df.sort_values('Giá/m² (triệu)')


    # out=f"./thuduc/BATDONGSAN_HCM_{datetime.now():%d%m%Y_%H%M}.csv" #cao


    # df.to_csv(out,index=False,encoding='utf-8-sig')
    # print(f"Đã lưu {len(df)} tin vào {out}")
    # return df

    out = f"./raw/tanphu.csv"

    # Nếu file chưa tồn tại → ghi mới, có header
    write_header = not os.path.exists(out)

    df.to_csv(out, index=False, encoding='utf-8-sig',
            mode='a', header=write_header)

    print(f"Đã thêm {len(df)} tin vào {out}")
    return df


async def main():
    base_url = "https://batdongsan.com.vn/nha-dat-ban-tan-phu/p{}"
    html_file_template = "./raw/tanphu/tanphu_p{}.html"
    for page_num in range(3, max_page+1):
        url = base_url.format(page_num)
        outfile = html_file_template.format(page_num)
        print(f"Đang cào trang {page_num}: {url}")
        await crawl_html(url, outfile)
        analyze_html(outfile)
        # (nếu muốn phân tích mỗi trang riêng hoặc gom lại sau)
    # nếu muốn phân tích tất cả sau đó: phân tích từ nhiều file hoặc từ một file lớn

if __name__=="__main__":
    max_page = 10  
    asyncio.run(main())